# Ch4：P0 / P2 / P8 / P6 分块归位 P4（四个独立 Jupyter Cell）

本 Notebook 基于原始 Ch4 Notebook，使用本次对话中**最新的位姿参数**。不再使用 `num`，也不会在定义函数时自动运行四块动作。

**执行顺序：**
1. 运行下方「参数和 IK」Cell。
2. 运行「公共动作函数」Cell（仅定义函数，**不会驱动硬件**）。
3. 按需**单独运行** P0、P2、P8、P6 对应的执行 Cell；运行哪个就只夹取哪一个。

| 运行 Cell | 抓取位置 | 放置目标 | 原索引 |
|---|---|---|---|
| P0 | P0 | P4 底层 | 0 |
| P2 | P2 | P4 第 2 层 | 1 |
| P8 | P8 | P4 第 3 层 | 2 |
| P6 | P6 | P4 顶层 | 3 |

**重要安全提醒：** P2 / P8 / P6 的放置高度仍是原程序的第 2 / 3 / 4 层高度。只运行这些 Cell **不会自动重新计算放置高度**，请确认 P4 已有对应的下层支撑，避免方块从高处掉落。每个执行 Cell 都会创建机械臂对象并执行真实运动；清空路径、准备停止措施后再运行。离线 IK 通过并不意味着整段路径不会碰撞。


## 1. 参数与 IK 检查函数

与原程序一致；使用本次对话提供的最新坐标。只定义参数与预检函数。

In [1]:
import math
import sys
from dataclasses import dataclass
from pathlib import Path
import numpy as np

# 此 Cell 只加载运动学，不创建串口或机械臂对象。
for sdk_dir in (Path.cwd(), Path.cwd() / '2.机械臂实验'):
    if (sdk_dir / 'arm5dof_kinematic.py').is_file():
        sys.path.insert(0, str(sdk_dir))
        break
else:
    raise FileNotFoundError('找不到 arm5dof_kinematic.py；请在课程目录打开本 Notebook。')

from arm5dof_kinematic import Arm5DoFKinematic


@dataclass(frozen=True)
class Pose:
    x: float
    y: float
    z: float
    pitch_deg: float
    roll_deg: float

    def xyz(self):
        return [self.x, self.y, self.z]


# 来自当前稳定 Ch3 的最新九点标定；P4 为中央回堆区。
CALIB_POINTS = {
    'P0': (240.4,  36.4, 0.278),
    'P2': (220.2, -62.1, 0.278),
    'P4': (177.5,  -9.0, 0.278),
    'P6': (140.1,  37.8, 0.278),
    'P8': (132.7, -58.4, 0.278),
}
CENTER_P4 = CALIB_POINTS['P4']
T_arm2ws = np.array([
    [0.999,  0.054, 0.0, 181.378],
    [-0.054, 0.999, 0.0, -11.644],
    [0.0,    0.0,   1.0,   0.278],
    [0.0,    0.0,   0.0,   1.0],
])

# Ch3 最新归位顺序为：最高层->P6，第二高->P8，第三层->P2，最低层->P0。
# Ch4 要由下至上回堆，因此下面严格使用反向物块顺序 P0、P2、P8、P6。
# 每项 source / source_above 均逐字复制自当前 Ch3 的 PLACE_POSES / PLACE_ABOVE_POSES。
SOURCE_POSES = (
    ('P0', Pose(248.4,  38.4, 30.278, 125.0, 179.2), Pose(236.4,  35.4, 85.278, 125.0, 179.2)),
    ('P2', Pose(235.2, -68.1, 30.278, 135.0, 179.2), Pose(220.2, -62.1, 85.278, 129.5, 179.2)),
    ('P8', Pose(145.7, -58.4, 20.278, 140.0, 179.2), Pose(132.7, -58.4, 75.278, 140.0, 179.2)),
    ('P6', Pose(155.1,  40.8, 20.278, 140.0, 179.2), Pose(136.1,  37.8, 75.278, 140.0, 179.2)),
)

# P4 目标：底层 -> 顶层，反向复制当前 Ch3 的 PICK_POSES / PICK_APPROACH_POSES。
# 第 2/3 层 Roll=92° 已在 Ch3 中验证，避免原始 83.7° 超出第五关节下限。
STACK_POSES = (
    Pose(198.2, -11.2,  33.0, 132.7, 144.5),  # P0 方块 -> P4 底层
    Pose(192.5, -11.3,  60.0, 125.0,  92.0),  # P2 方块 -> P4 第2层
    Pose(188.8, -15.0,  87.0, 120.8,  92.0),  # P8 方块 -> P4 第3层
    Pose(190.6,  -9.9, 112.0, 114.5, 144.5),  # P6 方块 -> P4 顶层
)
STACK_APPROACH_POSES = (
    Pose(198.2, -11.2,  79.5, 132.7, 144.5),
    Pose(193.6, -11.3,  79.5, 125.0,  92.0),
    Pose(189.8, -15.0,  92.0, 120.8,  92.0),
    Pose(190.6,  -9.9, 113.0, 114.5, 144.5),
)

MOVE_T = 0.8       # 沿用当前稳定 Ch3
GRIPPER_T = 0.5    # 沿用当前稳定 Ch3
START_BLOCK = 1
END_BLOCK = 4

# 与 set_tool_pose() 相同的 Z 补偿，确保离线预检与真机求解一致。
Z_ERROR_K, Z_ERROR_B = np.loadtxt(Path(sdk_dir) / 'config' / 'z_error_polyfit.txt', delimiter=',')


def sdk_adjusted_z(pose):
    return pose.z + float(Z_ERROR_K) * math.hypot(pose.x, pose.y) + float(Z_ERROR_B)


def validate_pose_ik(block, stage, pose):
    solutions = Arm5DoFKinematic().inverse_kinematic(
        pose.x, pose.y, sdk_adjusted_z(pose),
        math.radians(pose.pitch_deg), math.radians(pose.roll_deg),
    )
    if not solutions:
        raise RuntimeError(
            f'IK 预检失败：第 {block} 块 / {stage} / {pose}；'
            '停止：尚未创建机械臂对象。'
        )
    joints_deg = [round(math.degrees(theta), 2) for theta in solutions[-1]]
    print(f'IK OK  第 {block} 块 {stage:<13} -> {joints_deg}')
    return joints_deg


def validate_all_poses():
    if not (len(SOURCE_POSES) == len(STACK_POSES) == len(STACK_APPROACH_POSES) == 4):
        raise ValueError('Ch4 位姿表必须各有 4 项。')
    for index, (source_name, source, source_above) in enumerate(SOURCE_POSES):
        target = STACK_POSES[index]
        target_above = STACK_APPROACH_POSES[index]
        if target_above.z <= target.z:
            raise ValueError(f'第 {index + 1} 块 P4 上方点必须高于放置点。')
        validate_pose_ik(index + 1, f'Pick Above {source_name}', source_above)
        validate_pose_ik(index + 1, f'Pick {source_name}', source)
        validate_pose_ik(index + 1, 'Place Above P4', target_above)
        validate_pose_ik(index + 1, 'Place P4', target)
        validate_pose_ik(index + 1, 'Retreat P4', target_above)
    print('Ch4 全部 20 项关键姿态 IK 预检通过。')


def validate_parameters(start_block=START_BLOCK, end_block=END_BLOCK):
    if not (1 <= start_block <= end_block <= 4):
        raise ValueError('START_BLOCK / END_BLOCK 必须为 1~4，且起点不大于终点。')


## 2. 公共动作函数

定义 `move_to()`、`pick_and_stack()`、`run_single_block()`。运行本 Cell **不会执行机械臂动作**。

In [2]:
# 先运行参数 Cell。本 Cell 仅定义动作函数，不会自动连接或驱动机械臂。
def move_to(arm, pose, label):
    print(f'  {label:<14} -> ({pose.x:.1f}, {pose.y:.1f}, {pose.z:.1f}), '
          f'pitch={pose.pitch_deg:.1f}°, roll={pose.roll_deg:.1f}°')
    result = arm.set_tool_pose(
        pose.xyz(),
        tool_pitch=math.radians(pose.pitch_deg),
        tool_roll=math.radians(pose.roll_deg),
        T=MOVE_T, is_soft_move=True, is_wait=True,
    )
    if result is False:
        raise RuntimeError(f'{label} 运行时 IK 失败：{pose.xyz()}；立即停止。')


def pick_and_stack(arm, index):
    source_name, source, source_above = SOURCE_POSES[index]
    target, target_above = STACK_POSES[index], STACK_APPROACH_POSES[index]
    print(f'\n第 {index + 1} 块：{source_name} -> P4 第 {index + 1} 层')
    arm.gripper_open(T=GRIPPER_T, is_wait=True)
    move_to(arm, source_above, 'Pick Above')
    move_to(arm, source, 'Pick')
    arm.gripper_close(T=GRIPPER_T, is_wait=True)
    move_to(arm, source_above, 'Lift')
    arm.home()
    move_to(arm, target_above, 'Place Above')
    move_to(arm, target, 'Place P4')
    arm.gripper_open(angle=0.40,T=GRIPPER_T, is_wait=True)
    move_to(arm, target_above, 'Retreat')
    arm.home()


def run_single_block(index):
    """仅执行 SOURCE_POSES[index] 对应的方块；index: P0=0, P2=1, P8=2, P6=3。"""
    validate_parameters(index + 1, index + 1)
    # 保留原有完整的20项IK预检，且在创建硬件对象前完成
    validate_all_poses()

    from arm5dof_uservo import Arm5DoFUServo
    arm = Arm5DoFUServo()
    try:
        print(f"\\n单块归位：{SOURCE_POSES[index][0]} -> P4 第 {index + 1} 层")
        arm.home()
        pick_and_stack(arm, index)
        arm.gripper_open(T=GRIPPER_T, is_wait=True)
        arm.home()
        print(f"归位完成：{SOURCE_POSES[index][0]}")
    except Exception:
        # 无法确定机械臂的位置时，不再强制回Home或开夹爪
        print("运行中出现异常：已停止当前程序的后续动作。")
        print("请人工确认机械臂、方块和周围环境的状态后再恢复。")
        raise


## 3. 单独抓取 P0 → P4 第 1 层（底层）

仅运行下一个 Cell，即只执行 P0。

In [3]:
# 只执行 P0 -> P4 底层
run_single_block(0)

IK OK  第 1 块 Pick Above P0 -> [8.52, -73.18, 40.38, 67.8, -0.8]
IK OK  第 1 块 Pick P0       -> [8.79, -72.05, 82.77, 24.28, -0.8]
IK OK  第 1 块 Place Above P4 -> [-3.23, -83.15, 36.98, 88.88, -35.5]
IK OK  第 1 块 Place P4      -> [-3.23, -99.61, 92.83, 49.48, -35.5]
IK OK  第 1 块 Retreat P4    -> [-3.23, -83.15, 36.98, 88.88, -35.5]
IK OK  第 2 块 Pick Above P2 -> [-15.75, -60.08, 7.07, 92.5, -0.8]
IK OK  第 2 块 Pick P2       -> [-16.15, -60.25, 46.8, 58.45, -0.8]
IK OK  第 2 块 Place Above P4 -> [-3.34, -107.04, 76.39, 65.66, -88.0]
IK OK  第 2 块 Place P4      -> [-3.36, -113.42, 96.25, 52.17, -88.0]
IK OK  第 2 块 Retreat P4    -> [-3.34, -107.04, 76.39, 65.66, -88.0]
IK OK  第 3 块 Pick Above P8 -> [-23.75, -92.33, 24.77, 117.56, -0.8]
IK OK  第 3 块 Pick P8       -> [-21.84, -115.91, 98.73, 67.18, -0.8]
IK OK  第 3 块 Place Above P4 -> [-4.52, -113.26, 78.73, 65.33, -88.0]
IK OK  第 3 块 Place P4      -> [-4.54, -115.81, 84.38, 62.23, -88.0]
IK OK  第 3 块 Retreat P4    -> [-4.52, -113.26, 78.73, 65.33,

## 4. 单独抓取 P2 → P4 第 2 层

**先确保 P4 已有底层方块支撑**，否则不要运行此 Cell。

In [ ]:
# 只执行 P2 -> P4 第 2 层
run_single_block(1)

IK OK  第 1 块 Pick Above P0 -> [8.52, -73.18, 40.38, 67.8, -0.8]
IK OK  第 1 块 Pick P0       -> [8.79, -72.05, 82.77, 24.28, -0.8]
IK OK  第 1 块 Place Above P4 -> [-3.23, -83.15, 36.98, 88.88, -35.5]
IK OK  第 1 块 Place P4      -> [-3.23, -99.61, 92.83, 49.48, -35.5]
IK OK  第 1 块 Retreat P4    -> [-3.23, -83.15, 36.98, 88.88, -35.5]
IK OK  第 2 块 Pick Above P2 -> [-15.75, -60.08, 7.07, 92.5, -0.8]
IK OK  第 2 块 Pick P2       -> [-16.15, -60.25, 46.8, 58.45, -0.8]
IK OK  第 2 块 Place Above P4 -> [-3.34, -107.04, 76.39, 65.66, -88.0]
IK OK  第 2 块 Place P4      -> [-3.36, -113.42, 96.25, 52.17, -88.0]
IK OK  第 2 块 Retreat P4    -> [-3.34, -107.04, 76.39, 65.66, -88.0]
IK OK  第 3 块 Pick Above P8 -> [-23.75, -92.33, 24.77, 117.56, -0.8]
IK OK  第 3 块 Pick P8       -> [-21.84, -115.91, 98.73, 67.18, -0.8]
IK OK  第 3 块 Place Above P4 -> [-4.52, -113.26, 78.73, 65.33, -88.0]
IK OK  第 3 块 Place P4      -> [-4.54, -115.81, 84.38, 62.23, -88.0]
IK OK  第 3 块 Retreat P4    -> [-4.52, -113.26, 78.73, 65.33,

## 5. 单独抓取 P8 → P4 第 3 层

**先确保 P4 已有下方两层支撑**，否则不要运行此 Cell。

In [ ]:
# 只执行 P8 -> P4 第 3 层
run_single_block(2)

## 6. 单独抓取 P6 → P4 第 4 层（顶层）

**先确保 P4 已有下方三层支撑**，否则不要运行此 Cell。

In [ ]:
# 只执行 P6 -> P4 第 4 层
run_single_block(3)